# Relation of HHJ and TDNNS method
The TDNNS method for Reissner-Mindlin plates uses the HHJ elements to discretize the bending energy $\|\varepsilon(\beta)\|_{\mathbb{C},L^2}^2$ together with rotational degrees of freedom $\beta$. As gradient fields of $H^1(\Omega)$ are included in $H(\mathrm{curl})$ there is a strong relationship between these two models, especially for $t\to0$.

In [ ]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

E, nu, k = 10.92, 0.3, 5 / 6
G = E / (2 * (1 + nu))
# force
fz = -1


def CMatInv(mat, E, nu):
    return (
        12
        * (1 - nu**2)
        / E
        * (1 / (1 - nu) * mat - nu / (1 - nu**2) * Trace(mat) * Id(2))
    )


R = 5


def GenerateMesh(order, maxh=1):
    circ = Circle((0, 0), R).Face()
    circ.edges.name = "circ"
    shape = (
        circ
        - MoveTo(-R, -R).Rectangle(R, 2 * R).Face()
        - MoveTo(-R, -R).Rectangle(2 * R, R).Face()
    )
    shape.edges.Min(X).name = "left"
    shape.edges.Min(Y).name = "bottom"
    return Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=maxh * R / 3)).Curve(order)


mesh = GenerateMesh(2)
Draw(mesh)

In [ ]:
def ExactSolution(t=0.1, clamped=True, draw=False):
    r = sqrt(x**2 + y**2)
    xi = r / R
    Db = E * t**3 / (12 * (1 - nu**2))

    if clamped:
        # exact solution for clamped bc
        w_ex = (
            fz
            * t**3
            * R**4
            / (64 * Db)
            * (1 - xi**2)
            * ((1 - xi**2) + 8 * (t / R) ** 2 / (3 * k * (1 - nu)))
        )
    else:
        # exact solution for simply supported bc
        w_ex = (
            fz
            * t**3
            * R**4
            / (64 * Db)
            * (1 - xi**2)
            * (
                (6 + 2 * nu) / (1 + nu)
                - (1 + xi**2)
                + 8 * (t / R) ** 2 / (3 * k * (1 - nu))
            )
        )
    if draw:
        Draw(w_ex, mesh, "w_s_ex")
    return w_ex


ExactSolution(t=0.2, clamped=True, draw=True);

## TDNNS method for Reissner-Mindlin plate as hierarchical model on the top of HHJ
The TDNNS method for Reissner-Mindlin plates reads:

Find: $(\sigma,\beta, w) \in H(\mathrm{div} \mathrm{div})\times H(\mathrm{curl})\times H^1(\Omega)$ such that for all $(\tau,\delta, v) \in H(\mathrm{div} \mathrm{div})\times H(\mathrm{curl})\times H^1(\Omega)$
\begin{align}
\begin{array}{ccccll}
-\int_{\Omega} \mathbb{C}^{-1} \sigma : \tau\,dx & - & \left\langle \mathrm{div} \tau, \beta \right\rangle & = & 0 \\
-\left\langle \mathrm{div} \sigma, \delta \right\rangle & + & \frac{\kappa\, G}{t^2} \int_{\Omega} (\nabla w - \beta)\cdot(\nabla v - \delta)\,dx & = & \int_{\Omega} f v\,dx,
\end{array}
\end{align}
where
\begin{align}
\langle \mathrm{div} \boldsymbol{\sigma}, \beta\rangle &= \sum_{T\in\mathcal{T}}\int_T\mathrm{div}\boldsymbol{\sigma}\cdot\beta\,dx -\int_{\partial T}\boldsymbol{\sigma}_{nt}\beta_t\,ds=-\sum_{T\in\mathcal{T}}\int_T\boldsymbol{\sigma}:\nabla \beta\,dx +\int_{\partial T}\boldsymbol{\sigma}_{nn}\beta_n\,ds =-\langle \boldsymbol{\sigma}, \nabla \beta\rangle
\end{align}
and the inverted elasticity tensor (complience  tensor) $\mathbb{C}^{-1}\varepsilon = \frac{12(1-\nu^2)}{Et^3}\big(\frac{1}{1-\nu}\varepsilon-\frac{\nu}{1-\nu^2}\mathrm{tr}(\varepsilon)I_{2\times2}\big)$.

We can define the shear $\gamma=\nabla w-\beta\in H(\mathrm{curl})$ by a change of variables leading to:

Find: $(\sigma,\gamma, w) \in H(\mathrm{div} \mathrm{div})\times H(\mathrm{curl})\times H^1(\Omega)$ such that for all $(\tau,\delta, v) \in H(\mathrm{div} \mathrm{div})\times H(\mathrm{curl})\times H^1(\Omega)$
\begin{align}
\begin{array}{ccccll}
-\int_{\Omega} \mathbb{C}^{-1} \sigma : \tau\,dx & - & \left\langle \mathrm{div} \tau, \nabla w-\gamma \right\rangle & = & 0 \\
-\left\langle \mathrm{div} \sigma, \nabla v-\delta \right\rangle & + & \frac{\kappa\, G}{t^2} \int_{\Omega} \gamma\cdot\delta\,dx & = & \int_{\Omega} f v\,dx,
\end{array}
\end{align}

We recognise the term $\left\langle \mathrm{div} \tau, \nabla w \right\rangle$ appearing in the HHJ method. In fact, when setting $\gamma=0$ the HHJ method remains. This happens in the case $t\to 0$ forcing no-shearing, $\|\gamma\|_{L^2}^2=0$. Therefore, the TDNNS method for Reissner-Mindlin paltes can be seen as an hierarchical extension of the HHJ method for Kirchhoff-Love plates, converging to it in the limit of vanishing thickness. Note, that we could perform the change of variables $\gamma=\nabla w-\beta$ as $\nabla w$ and $\beta\in H(\mathrm{curl})$.

In [ ]:
def SolveRM_TDNNS(mesh, order=1, clamped=True, draw=False):
    if clamped:
        fesB = HCurl(mesh, order=order - 1, dirichlet="circ")
        fesS = HDivDiv(mesh, order=order - 1, dirichlet="")
    else:
        fesB = HCurl(mesh, order=order - 1)
        fesS = HDivDiv(mesh, order=order - 1, dirichlet="circ")
    fesW = H1(mesh, order=order, dirichlet="circ")

    fes = fesW * fesB * fesS
    (w, gamma, sigma), (v, delta, tau) = fes.TnT()

    n = specialcf.normal(2)

    a = BilinearForm(fes, symmetric=True)
    a += (
        -InnerProduct(CMatInv(sigma, E, nu), tau)
        + InnerProduct(tau, w.Operator("hesse") - grad(gamma))
        + InnerProduct(sigma, v.Operator("hesse") - grad(delta))
    ) * dx
    a += (
        -(sigma * n * n) * (Grad(v) - delta) * n - (tau * n * n) * (Grad(w) - gamma) * n
    ) * dx(element_boundary=True)
    a += k * G / t**2 * gamma * delta * dx

    f = LinearForm(fes)
    f += fz * v * dx

    gfsol = GridFunction(fes, autoupdate=True)
    gfw, gfbeta, gfsigma = gfsol.components

    with TaskManager():
        a.Assemble()
        f.Assemble()
        inv = a.mat.Inverse(fes.FreeDofs(), inverse="umfpack")
        gfsol.vec.data = inv * f.vec
    if draw:
        Draw(gfw, mesh, "w")

    return gfw, fes

In [ ]:
l = []
clamped = True

order = 1
with TaskManager():
    for t in [1, 0.1, 0.01]:
        l.append([])
        w_ex = ExactSolution(t=t, clamped=clamped)
        norm_w = sqrt(Integrate(w_ex * w_ex, mesh))

        for i in range(5):
            mesh = GenerateMesh(order=order, maxh=0.5**i)
            gfw, fes = SolveRM_TDNNS(mesh, order=order, clamped=clamped, draw=False)

            err = sqrt(Integrate((gfw - w_ex) * (gfw - w_ex), mesh)) / norm_w
            l[-1].append((fes.ndof, err))

In [ ]:
import matplotlib.pyplot as plt

plt.yscale("log")
plt.xscale("log")
plt.xlabel("ndof")
plt.ylabel("relative error")
for i in range(len(l)):
    ndof, err = zip(*l[i])
    plt.plot(ndof, err, "-*", label=f"t={10 ** (-i)}")
plt.legend()
plt.plot(
    ndof, [10 * dof ** (-2 / 2) for dof in ndof], "--", color="k", label="$O(h^2)$"
)
plt.show()

Next, we test if the Reissner-Mindlin solution converges to the Kirchhoff-Love limit solution for $t\to0$.

In [ ]:
def SolveKL_HHJ(mesh, order=1, clamped=True, draw=False):
    if clamped:
        fesS = HDivDiv(mesh, order=order - 1, dirichlet="")
    else:
        fesS = HDivDiv(mesh, order=order - 1, dirichlet="circ")
    fesW = H1(mesh, order=order, dirichlet="circ")

    fes = fesW * fesS
    (w, sigma), (v, tau) = fes.TnT()

    n = specialcf.normal(2)

    a = BilinearForm(fes, symmetric=True)
    a += (
        -InnerProduct(CMatInv(sigma, E, nu), tau)
        + InnerProduct(tau, w.Operator("hesse"))
        + InnerProduct(sigma, v.Operator("hesse"))
    ) * dx
    a += (-(sigma * n * n) * (Grad(v)) * n - (tau * n * n) * (Grad(w)) * n) * dx(
        element_boundary=True
    )

    f = LinearForm(fes)
    f += fz * v * dx

    gfsol = GridFunction(fes, autoupdate=True)
    gfw, gfsigma = gfsol.components

    with TaskManager():
        a.Assemble()
        f.Assemble()
        inv = a.mat.Inverse(fes.FreeDofs(), inverse="umfpack")
        gfsol.vec.data = inv * f.vec
    if draw:
        Draw(gfw, mesh, "w")

    return gfw, fes

In [ ]:
l = []
clamped = True
order = 1
mesh = GenerateMesh(order=order, maxh=0.5**4)

gfw_limit, fes = SolveKL_HHJ(mesh, order=order, clamped=clamped, draw=False)

with TaskManager():
    for t in [1, 0.1, 0.01, 0.001, 0.0001, 0.00001]:
        gfw, fes = SolveRM_TDNNS(mesh, order=order, clamped=clamped, draw=False)

        err = sqrt(Integrate((gfw - gfw_limit) * (gfw - gfw_limit), mesh))
        l.append((t, err))

In [ ]:
plt.yscale("log")
plt.xscale("log")
plt.xlabel("thicknes")
plt.ylabel("error")
t, err = zip(*l)
plt.plot(t, err, "-*", label="error")
plt.plot(t, [ts**2 for ts in t], "--", color="k", label="t^2")
plt.legend()
plt.show()